In [ ]:
"""Extract normalized lakehouse table metadata to OneLake as JSON."""

import json
from datetime import date, datetime


def _json_value(value):
    if isinstance(value, (date, datetime)):
        return value.isoformat()
    return value


StatementMeta(, 358f4f3f-de70-4196-8eb5-d28b6df1e7b6, 81, Finished, Available, Finished, False)

In [ ]:
# Mark this cell as the notebook parameter cell in Fabric.
tables_to_extract = "maag_silver.sales.order,maag_silver.sales.orderline,maag_silver.sales.orderpayment"
metadata_file_location = "Files/raw_metadata/metadata.json"


StatementMeta(, 358f4f3f-de70-4196-8eb5-d28b6df1e7b6, 148, Finished, Available, Finished, False)

In [ ]:
def _table_name(row):
    values = row.asDict(recursive=True)
    return values.get("tableName") or values.get("table_name") or values.get("name")


def _describe_table(table_name):
    rows = [row.asDict(recursive=True) for row in spark.sql(f"DESCRIBE EXTENDED {table_name}").collect()]
    columns = []
    properties = {}
    table_comment = None
    in_properties = False

    for row in rows:
        column_name = row.get("col_name")
        data_type = row.get("data_type")
        comment = row.get("comment")
        if not column_name:
            continue
        if column_name == "# Detailed Table Information":
            in_properties = True
            continue
        if column_name == "# Partition Information":
            in_properties = False
            continue
        if in_properties:
            key = str(column_name).strip()
            properties[key] = _json_value(data_type)
            if key.lower() in {"comment", "table_comment"}:
                table_comment = comment or data_type
            continue
        if str(column_name).startswith("#") or str(column_name).strip() == "":
            continue
        columns.append(
            {
                "name": str(column_name).strip(),
                "data_type": _json_value(data_type),
                "comment": comment,
                "nullable": True,
            }
        )

    catalog, schema, table = (table_name.split(".", 2) + [None, None])[:3]
    return {
        "catalog": catalog if table else None,
        "schema": schema if table else None,
        "table": table or table_name,
        "qualified_name": table_name,
        "table_comment": table_comment,
        "columns": columns,
        "properties": properties,
    }


requested_tables = [name.strip() for name in tables_to_extract.split(",") if name.strip()]
if not requested_tables:
    requested_tables = [
        _table_name(row)
        for row in spark.sql("SHOW TABLES").collect()
        if _table_name(row)
    ]

metadata = [_describe_table(table_name) for table_name in sorted(set(requested_tables))]
metadata_document = {
    "schema_version": "1.0",
    "source": "Microsoft Fabric Spark DESCRIBE EXTENDED",
    "generated_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "table_count": len(metadata),
    "tables": metadata,
}
print(f"Extracted {len(metadata)} tables")


StatementMeta(, 358f4f3f-de70-4196-8eb5-d28b6df1e7b6, 57, Finished, Available, Finished, False)

maag_silver.sales.order
maag_silver.sales.orderline
maag_silver.sales.orderpayment


In [ ]:
notebookutils.fs.put(
    metadata_file_location,
    json.dumps(metadata_document, ensure_ascii=False, indent=2),
    True,
)
run_summary = {
    "metadata_file_location": metadata_file_location,
    "table_count": metadata_document["table_count"],
    "schema_version": metadata_document["schema_version"],
}
print(json.dumps(run_summary))
notebookutils.notebook.exit(json.dumps(run_summary))


StatementMeta(, 358f4f3f-de70-4196-8eb5-d28b6df1e7b6, 149, Finished, Available, Finished, False)

True